In [ ]:
# Resolve defaults from this project's root, independently of the working directory.
from pathlib import Path as _KSPath
import sys as _ks_sys
from functools import partial as _ks_partial
_ks_start = _KSPath.cwd()
_ks_root = next((p for p in (_ks_start, *_ks_start.parents) if (p / 'kinspec_paths.py').is_file()), None)
if _ks_root is None:
    raise RuntimeError('Cannot locate the KinSPEC project root containing kinspec_paths.py. Start the notebook kernel from the project directory or a subdirectory.')
_ks_sys.path.insert(0, str(_ks_root))
from kinspec_paths import ROOT as KINSPEC_ROOT, project_path as _ks_project_path
_kp = _ks_partial(_ks_project_path, script_dir='others/baseline/TransformerCPI/script')
_ks_sys.path.insert(0, str(KINSPEC_ROOT / 'external/TransformerCPI'))
import os as _ks_os
_ks_os.chdir(KINSPEC_ROOT / 'others/baseline/TransformerCPI/script')



In [ ]:
import os
import sys
import glob
import gzip
import pickle
import time

import numpy as np
import pandas as pd
import torch
from torch.utils.data import IterableDataset, DataLoader
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc as sk_auc

# ============================================================
# User configuration: update the paths before running
# ============================================================

band = '10_50'

CONFIG = {
    # KIBA CSV path; the file must contain 'uniprot_id' and 'KIBA' columns
    "kiba_csv"      : _kp(f'dataset/KIBA_seen_LUDe_augment_sim08.csv'),

    # Glob pattern for KIBA pkl.gz shards
    "kiba_shards"   : _kp('dataset/preprocessed/TransformerCPI/KIBA_seen_LUDe_augment/KIBA_seen_augment_spectra_0.10_0.50_transformerCPI.part*.pkl.gz'),

    # Directory containing the five fold models
    "model_dir"     : _kp(f'model/TransformerCPI/SPECTRA_from_merged_{band}_transformerCPI'),

    # Model filename template; {fold} is replaced with 1 through 5
    "model_pattern" : "fold{fold}--lr=1e-4,dropout=0.1,weight_decay=1e-4,kernel=9,n_layer=3,batch=64,balance,lookaheadradam.pt",

    # Results directory
    "output_dir"    : _kp('others/baseline/TransformerCPI/results/KIBA_seen_LUDe_augment'),

    # Inference batch size
    "batch_size"    : 64,

    # Directory containing model.py; use None if it is already on sys.path
    "model_py_dir"  : _kp('external/TransformerCPI'),
}


# ============================================================
# Model hyperparameters must match training
# ============================================================
PROTEIN_DIM    = 100
ATOM_DIM       = 34
HID_DIM        = 64
N_LAYERS       = 3
N_HEADS        = 8
PF_DIM         = 256
DROPOUT        = 0.1
KERNEL_SIZE    = 9
BEDROC_ALPHA   = 80.5
KIBA_THRESHOLD = 3.0   # KIBA <= 3 → active (label=1)
MIN_SAMPLES    = 10    # Skip targets below the minimum sample count
EF_PERCENTAGES = (1, 2, 5, 10)

# ============================================================
# Add the model.py directory to the import path
# ============================================================
if CONFIG["model_py_dir"]:
    sys.path.insert(0, CONFIG["model_py_dir"])

from model import (Encoder, Decoder, Predictor,
                   DecoderLayer, SelfAttention,
                   PositionwiseFeedforward, pack)


# ============================================================
#  Dataset
# ============================================================
class ShardedPickleIterableAll(IterableDataset):
    """Traverse all shards in order, preserving CSV row order."""

    def __init__(self, shard_paths, device):
        self.shard_paths = list(shard_paths)
        self.device = device
        if not self.shard_paths:
            raise ValueError("No shard files found!")

    def __iter__(self):
        for p in self.shard_paths:
            with gzip.open(p, "rb") as f:
                data = pickle.load(f)
            for atom, adj, protein, label in data:
                yield (atom.float(), adj.float(), protein.float(), label.long())


def make_collate_fn(device):
    def _collate(batch):
        atoms, adjs, prots, labels = zip(*batch)
        return pack(list(atoms), list(adjs), list(prots), list(labels), device)
    return _collate


# ============================================================
# Build and load the model
# ============================================================
def build_model(device):
    encoder = Encoder(PROTEIN_DIM, HID_DIM, N_LAYERS, KERNEL_SIZE, DROPOUT, device)
    decoder = Decoder(ATOM_DIM, HID_DIM, N_LAYERS, N_HEADS, PF_DIM,
                      DecoderLayer, SelfAttention, PositionwiseFeedforward,
                      DROPOUT, device)
    return Predictor(encoder, decoder, device).to(device)


def load_model(ckpt_path, device):
    model = build_model(device)
    ckpt  = torch.load(ckpt_path, map_location=device)
    # A checkpoint may contain the complete training state or only state_dict
    if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
        state = ckpt["model_state_dict"]
    else:
        state = ckpt
    model.load_state_dict(state, strict=True)
    model.eval()
    print(f"  ✅ Loaded: {os.path.basename(ckpt_path)}")
    return model


# ============================================================
# Inference
# ============================================================
def run_inference(model, loader):
    """Return (y_true: np.ndarray[int], y_score: np.ndarray[float])."""
    y_true, y_score = [], []
    with torch.no_grad():
        for data_pack in loader:
            labels, _, scores = model(data_pack, train=False)
            y_true.extend(labels)
            y_score.extend(scores)
    return np.array(y_true, dtype=int), np.array(y_score, dtype=float)


# ============================================================
# Metric calculation
# ============================================================
def calculate_bedroc(y_true, y_score, alpha=BEDROC_ALPHA):
    y_true  = np.asarray(y_true, dtype=int)
    y_score = np.asarray(y_score, dtype=float)
    N     = len(y_true)
    n_act = int(np.sum(y_true == 1))
    if n_act == 0:
        return 0.0

    order  = np.argsort(y_score)[::-1]
    ranks  = np.arange(1, N + 1)[y_true[order] == 1]
    x      = (ranks - 0.5) / N

    C       = (1.0 - np.exp(-alpha)) / alpha
    rie     = np.mean(np.exp(-alpha * x)) / C

    x_max   = (np.arange(1, n_act + 1) - 0.5) / N
    x_min   = (np.arange(N - n_act + 1, N + 1) - 0.5) / N
    rie_max = np.mean(np.exp(-alpha * x_max)) / C
    rie_min = np.mean(np.exp(-alpha * x_min)) / C

    denom = rie_max - rie_min
    return float((rie - rie_min) / denom) if denom > 0 else 0.0


def calculate_ef(y_true, y_score, percentages=EF_PERCENTAGES):
    y_true  = np.asarray(y_true, dtype=int)
    y_score = np.asarray(y_score, dtype=float)
    N        = len(y_true)
    n_active = int(np.sum(y_true == 1))
    base_ratio = n_active / N if N > 0 else 0.0

    order         = np.argsort(y_score)[::-1]
    sorted_labels = y_true[order]

    results = {}
    for p in percentages:
        if n_active == 0 or base_ratio == 0:
            results[p] = 0.0
            continue
        n_sel     = max(1, int(round(N * p / 100.0)))
        n_act_top = int(np.sum(sorted_labels[:n_sel] == 1))
        results[p] = (n_act_top / n_sel) / base_ratio
    return results


def compute_metrics(y_true, y_score):
    """Calculate all metrics for one evaluation and return a dictionary."""
    metrics = {}
    if len(set(y_true)) < 2:
        metrics["ROC_AUC"] = float("nan")
        metrics["PR_AUC"]  = float("nan")
    else:
        metrics["ROC_AUC"] = float(roc_auc_score(y_true, y_score))
        prec, rec, _ = precision_recall_curve(y_true, y_score)
        metrics["PR_AUC"] = float(sk_auc(rec, prec))

    metrics["BEDROC"] = calculate_bedroc(y_true, y_score)

    for p, v in calculate_ef(y_true, y_score).items():
        metrics[f"EF_{p}%"] = v

    return metrics


def aggregate_fold_metrics(fold_metrics_list):
    """Aggregate fold metrics into {metric_mean, metric_std}."""
    keys = fold_metrics_list[0].keys()
    agg  = {}
    for k in keys:
        vals = [m[k] for m in fold_metrics_list if not np.isnan(m[k])]
        agg[f"{k}_mean"] = float(np.mean(vals)) if vals else float("nan")
        agg[f"{k}_std"]  = float(np.std(vals))  if vals else float("nan")
    return agg


# ============================================================
# Main workflow
# ============================================================
def main():
    os.makedirs(CONFIG["output_dir"], exist_ok=True)
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    # ----------------------------------------------------------
    # 1. Load the CSV
    # ----------------------------------------------------------
    print(f"\n📂 Loading KIBA CSV: {CONFIG['kiba_csv']}")
    df = pd.read_csv(CONFIG["kiba_csv"])

    for col in ("uniprot_id", "KIBA"):
        if col not in df.columns:
            raise ValueError(f"CSV missing required column '{col}'. "
                             f"Available: {list(df.columns)}")

    df["active"] = (df["KIBA"] <= KIBA_THRESHOLD).astype(int)
    uniprot_ids  = df["uniprot_id"].values       # shape (N,)
    csv_labels   = df["active"].values           # shape (N,)
    n_csv        = len(df)
    print(f"  Total samples    : {n_csv}")
    print(f"  Actives (KIBA≤3) : {csv_labels.sum()} ({csv_labels.mean()*100:.1f}%)")

    # ----------------------------------------------------------
    # 2. Read shards in CSV row order
    # ----------------------------------------------------------
    shard_paths = sorted(glob.glob(CONFIG["kiba_shards"]))
    if not shard_paths:
        raise FileNotFoundError(f"No shard files matched: {CONFIG['kiba_shards']}")
    print(f"\n📦 Found {len(shard_paths)} shard file(s)")

    # Verify that the total sample count matches the CSV
    total_pkl = 0
    for p in shard_paths:
        with gzip.open(p, "rb") as f:
            total_pkl += len(pickle.load(f))
    if total_pkl != n_csv:
        print(f"⚠️  WARNING: CSV rows ({n_csv}) != pkl samples ({total_pkl}). "
              f"Per-target alignment may be wrong!")
    else:
        print(f"  ✅ Sample count match: {total_pkl}")

    # ----------------------------------------------------------
    # 3. Construct the full-dataset DataLoader; shuffle=False preserves CSV order
    # ----------------------------------------------------------
    dataset = ShardedPickleIterableAll(shard_paths, device)
    loader  = DataLoader(dataset, batch_size=CONFIG["batch_size"],
                         shuffle=False, num_workers=0,
                         collate_fn=make_collate_fn(device))

    # ----------------------------------------------------------
    # 4. Run inference for each fold
    # ----------------------------------------------------------
    fold_scores = []   # List[np.ndarray] containing prediction scores for each fold
    fold_labels = None # Record y_true once; it should be identical across folds

    for fold in range(1, 6):
        ckpt = os.path.join(CONFIG["model_dir"],
                            CONFIG["model_pattern"].format(fold=fold))
        if not os.path.exists(ckpt):
            raise FileNotFoundError(f"Checkpoint not found: {ckpt}")

        print(f"\n🔍 Fold {fold}/5")
        t0    = time.time()
        model = load_model(ckpt, device)

        y_true, y_score = run_inference(model, loader)
        print(f"  Inference: {time.time()-t0:.1f}s | "
              f"samples={len(y_true)} | actives={y_true.sum()}")

        if not np.array_equal(y_true, csv_labels):
            print("  ⚠️  Label mismatch between pkl and CSV! Using pkl labels.")

        if fold_labels is None:
            fold_labels = y_true

        del model
        if device.type == "cuda":
            torch.cuda.empty_cache()

        fold_scores.append(y_score)

    fold_scores  = np.stack(fold_scores, axis=0)   # (5, N)
    y_true_final = fold_labels                      # (N,)

    # ----------------------------------------------------------
    # 5. Overall evaluation
    # ----------------------------------------------------------
    print("\n" + "="*60)
    print("📊 Computing OVERALL metrics...")

    n_total         = len(y_true_final)
    n_actives       = int(y_true_final.sum())
    pr_auc_baseline = n_actives / n_total

    overall_fold_metrics = []
    overall_fold_rows    = []

    for fold_idx in range(5):
        m = compute_metrics(y_true_final, fold_scores[fold_idx])
        overall_fold_metrics.append(m)
        overall_fold_rows.append({
            "Fold"              : fold_idx + 1,
            "N"                 : n_total,
            "Actives"           : n_actives,
            "ActiveRatio"       : round(n_actives / n_total, 4),
            "ROC_AUC"           : round(m["ROC_AUC"],  4),
            "PR_AUC"            : round(m["PR_AUC"],   4),
            "PR_AUC - Baseline" : round(m["PR_AUC"] - pr_auc_baseline, 4),
            "BEDROC"            : round(m["BEDROC"],   4),
            "EF_1%"             : round(m["EF_1%"],    3),
            "EF_2%"             : round(m["EF_2%"],    3),
            "EF_5%"             : round(m["EF_5%"],    3),
            "EF_10%"            : round(m["EF_10%"],   3),
        })

    overall_agg = aggregate_fold_metrics(overall_fold_metrics)

    print(f"  N={n_total}, Actives={n_actives} ({n_actives/n_total*100:.1f}%), "
          f"PR Baseline={pr_auc_baseline:.4f}")
    print(f"  ROC-AUC : {overall_agg['ROC_AUC_mean']:.4f} ± {overall_agg['ROC_AUC_std']:.4f}")
    print(f"  PR-AUC  : {overall_agg['PR_AUC_mean']:.4f} ± {overall_agg['PR_AUC_std']:.4f}")
    print(f"  BEDROC  : {overall_agg['BEDROC_mean']:.4f} ± {overall_agg['BEDROC_std']:.4f}")
    for p in EF_PERCENTAGES:
        print(f"  EF@{p:2d}%  : {overall_agg[f'EF_{p}%_mean']:.3f} ± "
              f"{overall_agg[f'EF_{p}%_std']:.3f}")

    # ----------------------------------------------------------
    # 6. Per-target evaluation
    # ----------------------------------------------------------
    print("\n" + "="*60)
    print("📊 Computing PER-TARGET metrics...")

    targets                 = np.unique(uniprot_ids)
    per_target_summary_rows = []
    per_target_fold_rows    = []
    skipped                 = 0

    for tgt in targets:
        mask  = (uniprot_ids == tgt)
        yt    = y_true_final[mask]
        n     = int(mask.sum())
        n_act = int(yt.sum())

        # Skip small targets and single-class targets, for which AUC/BEDROC cannot be calculated
        if n < MIN_SAMPLES or n_act == 0 or n_act == n:
            skipped += 1
            continue

        pr_baseline_tgt  = n_act / n
        tgt_fold_metrics = []

        for fold_idx in range(5):
            ys = fold_scores[fold_idx][mask]
            m  = compute_metrics(yt, ys)
            tgt_fold_metrics.append(m)

            per_target_fold_rows.append({
                "Target"            : tgt,
                "Fold"              : fold_idx + 1,
                "N"                 : n,
                "Actives"           : n_act,
                "ActiveRatio"       : round(n_act / n, 4),
                "ROC_AUC"           : round(m["ROC_AUC"],  4),
                "PR_AUC"            : round(m["PR_AUC"],   4),
                "PR_AUC - Baseline" : round(m["PR_AUC"] - pr_baseline_tgt, 4),
                "BEDROC"            : round(m["BEDROC"],   4),
                "EF_1%"             : round(m["EF_1%"],    3),
                "EF_2%"             : round(m["EF_2%"],    3),
                "EF_5%"             : round(m["EF_5%"],    3),
                "EF_10%"            : round(m["EF_10%"],   3),
            })

        agg = aggregate_fold_metrics(tgt_fold_metrics)
        per_target_summary_rows.append({
            "Target"                 : tgt,
            "N"                      : n,
            "Actives"                : n_act,
            "ActiveRatio"            : round(n_act / n, 4),
            "ROC_AUC_mean"           : round(agg["ROC_AUC_mean"],  4),
            "ROC_AUC_std"            : round(agg["ROC_AUC_std"],   4),
            "PR_AUC_mean"            : round(agg["PR_AUC_mean"],   4),
            "PR_AUC_std"             : round(agg["PR_AUC_std"],    4),
            "PR_AUC_mean - Baseline" : round(agg["PR_AUC_mean"] - pr_baseline_tgt, 4),
            "BEDROC_mean"            : round(agg["BEDROC_mean"],   4),
            "BEDROC_std"             : round(agg["BEDROC_std"],    4),
            "EF_1%_mean"             : round(agg["EF_1%_mean"],    3),
            "EF_1%_std"              : round(agg["EF_1%_std"],     3),
            "EF_2%_mean"             : round(agg["EF_2%_mean"],    3),
            "EF_2%_std"              : round(agg["EF_2%_std"],     3),
            "EF_5%_mean"             : round(agg["EF_5%_mean"],    3),
            "EF_5%_std"              : round(agg["EF_5%_std"],     3),
            "EF_10%_mean"            : round(agg["EF_10%_mean"],   3),
            "EF_10%_std"             : round(agg["EF_10%_std"],    3),
        })

    per_target_summary_df = (pd.DataFrame(per_target_summary_rows)
                               .sort_values("Target")
                               .reset_index(drop=True))
    per_target_fold_df    = (pd.DataFrame(per_target_fold_rows)
                               .sort_values(["Target", "Fold"])
                               .reset_index(drop=True))

    print(f"  Evaluated {len(per_target_summary_df)} targets  "
          f"(skipped {skipped}: n<{MIN_SAMPLES} or single class)")

    # ----------------------------------------------------------
    # 7. Save results
    # ----------------------------------------------------------

    # 7a. Overall numeric metrics
    overall_summary_csv = os.path.join(CONFIG["output_dir"], f"kiba_{band}_overall_summary.csv")
    pd.DataFrame([{
        "N": n_total,
        "Actives": n_actives,
        "ActiveRatio": n_actives / n_total,
        "PR_baseline": pr_auc_baseline,
        "PR_AUC_minus_baseline": overall_agg["PR_AUC_mean"] - pr_auc_baseline,
        **overall_agg,
    }]).to_csv(overall_summary_csv, index=False)
    print(f"\nSaved metrics to {overall_summary_csv}")

    # 7b. Overall per-fold CSV
    overall_fold_csv = os.path.join(CONFIG["output_dir"], f"kiba_{band}_overall_all_folds.csv")
    pd.DataFrame(overall_fold_rows).to_csv(overall_fold_csv, index=False)
    print(f"💾 {overall_fold_csv}")

    # 7c. Per-target mean±std CSV + TSV
    per_target_summary_csv = os.path.join(CONFIG["output_dir"], f"kiba_{band}_per_target_summary.csv")
    # per_target_summary_tsv = os.path.join(CONFIG["output_dir"], "kiba_per_target_summary.tsv")
    per_target_summary_df.to_csv(per_target_summary_csv, index=False)
    # per_target_summary_df.to_csv(per_target_summary_tsv, index=False, sep="\t")
    print(f"💾 {per_target_summary_csv}")
    # print(f"💾 {per_target_summary_tsv}")

    # 7d. Per-target per-fold CSV
    per_target_fold_csv = os.path.join(CONFIG["output_dir"], f"kiba_{band}_per_target_all_folds.csv")
    per_target_fold_df.to_csv(per_target_fold_csv, index=False)
    print(f"💾 {per_target_fold_csv}")

    # Preview
    print("\n📋 Per-target summary (first 5 rows):")
    print(per_target_summary_df.head(5).to_string(index=False))

    print("\n✅ All done!")


if __name__ == "__main__":
    main()

Device: cuda:0

📂 Loading KIBA CSV: /home/s1856122/dataset/KIBA/SPECTRA_comparison_from_merged/KIBA_10_50/KIBA_10_50_from_merged_augment/KIBA_SPECTRA_VSDS_10_50_from_merged_remove_overlap_data_t0.80_cleavage_spectra_unseen_target_mol_deepcoy_augment.csv
  Total samples    : 20956
  Actives (KIBA≤3) : 856 (4.1%)

📦 Found 5 shard file(s)
  ✅ Sample count match: 20956

🔍 Fold 1/5
  ✅ Loaded: fold1--lr=1e-4,dropout=0.1,weight_decay=1e-4,kernel=9,n_layer=3,batch=64,balance,lookaheadradam.pt


/tmp/ipykernel_1311092/3536036007.py:111: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt  = torch.load(ckpt_path, map_location=device)


  Inference: 19.8s | samples=20956 | actives=856

🔍 Fold 2/5
  ✅ Loaded: fold2--lr=1e-4,dropout=0.1,weight_decay=1e-4,kernel=9,n_layer=3,batch=64,balance,lookaheadradam.pt
  Inference: 19.8s | samples=20956 | actives=856

🔍 Fold 3/5
  ✅ Loaded: fold3--lr=1e-4,dropout=0.1,weight_decay=1e-4,kernel=9,n_layer=3,batch=64,balance,lookaheadradam.pt
  Inference: 19.8s | samples=20956 | actives=856

🔍 Fold 4/5
  ✅ Loaded: fold4--lr=1e-4,dropout=0.1,weight_decay=1e-4,kernel=9,n_layer=3,batch=64,balance,lookaheadradam.pt
  Inference: 19.9s | samples=20956 | actives=856

🔍 Fold 5/5
  ✅ Loaded: fold5--lr=1e-4,dropout=0.1,weight_decay=1e-4,kernel=9,n_layer=3,batch=64,balance,lookaheadradam.pt
  Inference: 19.7s | samples=20956 | actives=856

📊 Computing OVERALL metrics...
  N=20956, Actives=856 (4.1%), PR Baseline=0.0408
  ROC-AUC : 0.5419 ± 0.1053
  PR-AUC  : 0.0479 ± 0.0198
  BEDROC  : 0.0316 ± 0.0455
  EF@ 1%  : 0.723 ± 1.102
  EF@ 2%  : 0.689 ± 1.064
  EF@ 5%  : 0.818 ± 1.055
  EF@10%  : 0.832 ±